#### Importing required libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#### Data Loading

In this section, we load the required datasets for the analysis.

For the initial investigation, we start with the Orders dataset because it contains the delivery-related timestamps required to identify delayed orders.

In [4]:
orders=pd.read_csv("data/olist_orders_dataset.csv")
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


#### Initial Dataset Inspection
The goal of this step is to understand structure of the data,identifying important columns,inspect data types on the high level and to identify missing values if any.

In [5]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


#### Observations
- The dataset contains 99441 entries

#### Data cleaning

In [49]:
orders[['order_delivered_customer_date','order_estimated_delivery_date']]=orders[['order_delivered_customer_date','order_estimated_delivery_date']].apply(pd.to_datetime,axis=0)

In [54]:
orders[["order_delivered_customer_date","order_estimated_delivery_date"]].dtypes

order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

#### Handling missing values

In [56]:
orders["order_delivered_customer_date"].isnull().sum()

np.int64(2965)

#### conclusion
- There are 2000 missing entries in the order_delivered_customer_date field.These likely represent specific order statuses, such as processed, shipped, cancelled, invoiced, or unrecorded orders.
- To determine whether an order was delivered on time or delayed, both the actual delivery date and the estimated delivery date must be available.
- Orders with missing delivery dates cannot be reliably classified as delayed or on-time. Therefore, these records will be excluded from the delivery delay analysis.

In [68]:
orders=orders.dropna(subset=["order_delivered_customer_date"])
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


#### conclusion
- After removing orders with missing delivery dates, 96,476 orders remained for analysis.
- These records contain both actual and estimated delivery dates, allowing delivery performance to be evaluated accurately.

#### Feature Engineering: Delayed Order Flag
To evaluate delivery performance, a new feature called `is_delayed` is created.

Business Rule:
- If the actual delivery date is later than the estimated delivery date, the order is classified as delayed.
-  Otherwise, the order is classified as on-time.

This feature converts raw delivery timestamps into a business-friendly metric that can be used throughout the analysis.


In [74]:
orders["is_delayed"]=orders["order_delivered_customer_date"]>orders["order_estimated_delivery_date"]

In [76]:
orders['is_delayed'].value_counts()

is_delayed
False    88649
True      7827
Name: count, dtype: int64

#### Observation

After creating the delayed order flag, the majority of orders were delivered on time.

However, 7,827 orders were delivered later than the estimated delivery date. To understand the scale of this issue, the delayed order percentage should be calculated rather than relying on raw counts alone.

#### KPI 1: Percentage of Delayed Orders

The delayed order rate measures the proportion of delivered orders that arrived later than the estimated delivery date.

Formula:

(Delayed Orders / Total Orders) × 100

This KPI helps quantify the scale of delivery performance issues within the marketplace.

In [99]:
(orders['is_delayed'].value_counts(normalize=True)*100).to_frame()

,proportion
is_delayed,
False,91.887101
True,8.112899


#### KPI Observation

Approximately 8% of delivered orders arrived later than the estimated delivery date, while around 92% were delivered on time.

This indicates that the overall delivery performance is relatively strong. However, delivery delays still affect a meaningful number of customers. Further analysis is required to determine whether delayed deliveries are associated with lower customer satisfaction and review scores.
